# Step 4. Benchmark question generation

In [3]:
import os
import pandas as pd
from typing import Any, Optional, Type
from pydantic import BaseModel
import pickle

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage
from langchain_core.prompts import PromptTemplate

import asyncio

In [4]:
SPLITS_PATH = './splits'
DATA_PATH = './data'
BASE_URL = 'http://192.168.1.42:11434/'  # 'http://localhost:11434/'
OLLAMA_MODEL = 'gemma3:27b-it-qat'  # 'gemma4:31b'  # 'qwen3:14b'

In [5]:
with open(os.path.join(SPLITS_PATH, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)

In [6]:
splits[0].metadata

{'title': 'Уильям Гибсон. Виртуальный свет',
 'chapter': '2. "ГРОМИЛА" В ДЕЛЕ',
 'section': 'Машины "Интенсекьюра"...',
 'id': 'ec656dca4ba1380173170a166b24d9cb',
 'size': 12329,
 'collection': 'Virtual_Light'}

In [7]:
splits[1].metadata

{'title': 'Уильям Гибсон. Виртуальный свет',
 'chapter': '2. "ГРОМИЛА" В ДЕЛЕ',
 'section': 'Что это у тебя за штука...',
 'id': 'd56a86626329914a0590d6f3913f34b6',
 'size': 2541,
 'collection': 'Virtual_Light'}

### Define and test the LLM

In [8]:
# OLLAMA_CONTEXT_LENGTH environment variable needs to be set to 32000
# set OLLAMA_CONTEXT_LENGTH=32000

In [9]:
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0.5,
    base_url=BASE_URL,
)

In [10]:
%%time
llm.invoke('Привет')

CPU times: total: 15.6 ms
Wall time: 3.09 s


AIMessage(content='Привет! Чем могу помочь?\n', additional_kwargs={}, response_metadata={'model': 'gemma3:27b-it-qat', 'created_at': '2026-04-10T07:14:31.5161718Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2637091600, 'load_duration': 85130600, 'prompt_eval_count': 28, 'prompt_eval_duration': 753083700, 'eval_count': 14, 'eval_duration': 1782410400, 'model_name': 'gemma3:27b-it-qat'}, id='run--105f0ef5-1b87-4891-973f-cc21a4fee38c-0', usage_metadata={'input_tokens': 28, 'output_tokens': 14, 'total_tokens': 42})

### Add structured output

In [11]:
class QuestionGenerationSchema(BaseModel):
    number_of_questions: int
    questions: list[str]

In [12]:
def query_generation(system_prompt: str, md_text: str) -> QuestionGenerationSchema:
    content = f'\n\nФрагмент текста:\n\n{md_text}'
    try:
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": content
            },
        ]
        structured_llm = llm.with_structured_output(QuestionGenerationSchema)
        response = structured_llm.invoke(messages)
        if not isinstance(response, QuestionGenerationSchema):
            raise RuntimeError("LLM did not return correct scheme response")
        return response
    except Exception as ex:
        print(f"Question generation error: {ex}")

### Prompt

In [13]:
QUESTION_GEN_PROMPT = '''
Твоя задача - придумать от {min_questions} до {max_questions} вопросов, опираясь на фрагмент художественного текста.
Формулируй вопросы так как если бы они задавались на викторине.
Убедись, что на вопросы можно ответить, опираясь на фрагмент текста.

В формулировке вопроса необходимо и очень **важно** использовать ключевые слова из текста:
- имена персонажей
- названия мест и организаций
- упомянутые в тексте предметы и гаджеты.

Правила подготовки вопросов:
- Вопросы должны быть заданы на русском языке.
- Вопросы не должны повторяться.
- В вопросе **нельзя** ссылаться на фрагмент текста,
**нельзя** использовать формулировки "согласно описанию в тексте", "согласно тексту", "как указано в тексте", "упоминается в фрагменте текста"

Вывод должен быть строго в формате JSON:
"number_of_questions": int, // Сколько вопросов ты подготовил для данного фрагмента текста.
"questions": [str, ..., str] // Список вопросов.
'''

In [14]:
prompt_template = PromptTemplate.from_template(QUESTION_GEN_PROMPT)
system_prompt = prompt_template.invoke({
    'min_questions': 1,
    'max_questions': 4,}).text

### Generate and save questions

In [15]:
%%time
result = pd.DataFrame()
for i in range(1, len(splits)):  # without toc chunk
    md_text = f'{splits[i].page_content}'
    metadata = splits[i].metadata
    chapter_name = metadata.get('chapter')
    section_name = metadata.get('section')
    chunk_id = metadata.get('id')
   
    response = query_generation(system_prompt=system_prompt, md_text=md_text)
    print(f'Фрагмент: {i+1}')
    print(f'Глава: {chapter_name}')
    print(f'Раздел: {section_name}')
    print(f'Кол-во вопросов: {response.number_of_questions}')
    for q in response.questions:
        print(f'- {q}')
        df_row = pd.DataFrame(data=[{
            'question': q,
            'chunk_id': chunk_id,
            'section_name': section_name,
            'chapter_name': chapter_name,
        }])
        result = pd.concat([result, df_row], ignore_index=True)
    print()

Фрагмент: 2
Глава: 2. "ГРОМИЛА" В ДЕЛЕ
Раздел: Что это у тебя за штука...
Кол-во вопросов: 4
- Что, по словам Терви, является его "ружьем" и чем оно стреляет?
- Какой предмет использовал Терви для питания своего необычного оружия?
- Какое телевизионное шоу любили смотреть Райделл и его отец?
- Как зовут женщину, которая пришла к Райделлу в больнице и какую должность она занимает?

Фрагмент: 3
Глава: 2. "ГРОМИЛА" В ДЕЛЕ
Раздел: Райделл вставил ключ...
Кол-во вопросов: 4
- Какую функцию выполняли камеры, установленные под задним бампером автомобиля "Громила", по мнению Райделла?
- Какое неофициальное название носил спутник, который сотрудники "Интенсекьюра" старались не упоминать вслух, и как он назывался официально?
- Что Саблетт использовал для прослушивания связи, учитывая помехи в стенах автомойки?
- Какие предметы Саблетт надевал перед вызовом и зачем, и какую реакцию это вызвало у Райделла?

Фрагмент: 4
Глава: 2. "ГРОМИЛА" В ДЕЛЕ
Раздел: В тот раз пристрелив...
Кол-во вопросов: 4
-

In [17]:
print(f'Generated questions: {result.shape[0]}')

Generated questions: 382


In [18]:
result.head(10)

,question,chunk_id,section_name,chapter_name
0,"Что, по словам Терви, является его ""ружьем"" и ...",d56a86626329914a0590d6f3913f34b6,Что это у тебя за штука...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
1,Какой предмет использовал Терви для питания св...,d56a86626329914a0590d6f3913f34b6,Что это у тебя за штука...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
2,Какое телевизионное шоу любили смотреть Райдел...,d56a86626329914a0590d6f3913f34b6,Что это у тебя за штука...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
3,"Как зовут женщину, которая пришла к Райделлу в...",d56a86626329914a0590d6f3913f34b6,Что это у тебя за штука...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
4,"Какую функцию выполняли камеры, установленные ...",6851dc51ddcc613e44670dfc2c6ec6f3,Райделл вставил ключ...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
5,"Какое неофициальное название носил спутник, ко...",6851dc51ddcc613e44670dfc2c6ec6f3,Райделл вставил ключ...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
6,Что Саблетт использовал для прослушивания связ...,6851dc51ddcc613e44670dfc2c6ec6f3,Райделл вставил ключ...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
7,Какие предметы Саблетт надевал перед вызовом и...,6851dc51ddcc613e44670dfc2c6ec6f3,Райделл вставил ключ...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
8,Какую визитную карточку представил Райделлу Ве...,c82554e85771eb7f04616353c5666091,В тот раз пристрелив...,"2. ""ГРОМИЛА"" В ДЕЛЕ"
9,Какой предмет приносила Карен Мендельсон Райде...,c82554e85771eb7f04616353c5666091,В тот раз пристрелив...,"2. ""ГРОМИЛА"" В ДЕЛЕ"


In [19]:
min(result.question.str.len())

39

In [20]:
result.to_csv(os.path.join(DATA_PATH, 'questions.csv'), sep=';', index=False)